In [1]:
import pandas as pd
import numpy as np

## Series Indexing

`[]` on a Series works like NumPy array indexing, but also accepts label names.

- Single label or integer → returns a scalar.
- List of labels or integers → returns a Series.
- Slice → returns a Series.
- Boolean array → returns filtered Series.

### Prefer `loc` Over `[]` for Label Selection

`loc` is the preferred way to select by label because `[]` behaves inconsistently when the index contains integers:

- Integer index → `[]` treats integers as **labels**.
- String index → `[]` treats integers as **positions**.

This ambiguity can cause silent bugs. `loc` always uses **labels**; `iloc` always uses **integer positions**.

| Operator | Indexes by | Works when |
|----------|-----------|------------|
| `[]` | Label or position (ambiguous) | Simple cases only |
| `loc` | Label exclusively | Always safe for labels |
| `iloc` | Integer position exclusively | Always safe for positions |

### Label Slicing with `loc`

Unlike regular Python slicing, label-based slicing with `loc` is **endpoint inclusive**:

```python
obj.loc["b":"c"]  # includes both "b" and "c"
```

### Assignment with `loc`

Assigning to a `loc` slice modifies the corresponding section of the Series in place.

## Key Points

- `[]` accepts labels, integers, slices, and boolean arrays.
- `loc` indexes exclusively by label — always prefer it over `[]` for label selection.
- `iloc` indexes exclusively by integer position — use it for position-based access.
- Label slicing with `loc` is **inclusive** of the endpoint.
- `loc` and `iloc` use square bracket notation, not function call notation.

In [2]:
obj = pd.Series(np.arange(4.), index=["a", "b", "c", "d"])

obj

a    0.0
b    1.0
c    2.0
d    3.0
dtype: float64

In [3]:
obj["b"]

np.float64(1.0)

In [4]:
obj[1]

/tmp/ipykernel_18440/2469632899.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  obj[1]


np.float64(1.0)

In [5]:
obj[2:4]

c    2.0
d    3.0
dtype: float64

In [6]:
obj[["b", "a", "d"]]

b    1.0
a    0.0
d    3.0
dtype: float64

In [7]:
obj[[1, 3]]

/tmp/ipykernel_18440/2982346117.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  obj[[1, 3]]


b    1.0
d    3.0
dtype: float64

In [8]:
obj[obj < 2]

a    0.0
b    1.0
dtype: float64

In [41]:
obj.loc[["b", "a", "d"]]
# obj.loc["b", "a", "d"] wont work because passes a tuple that is taken as a label

b    1.0
a    0.0
d    3.0
dtype: float64

In [10]:
# Integer index: [] treats integers as labels
obj1 = pd.Series([1, 2, 3], index=[2, 0, 1])

# String index: [] treats integers as positions
obj2 = pd.Series([1, 2, 3], index=["a", "b", "c"])

print(obj1[[0, 1, 2]])  # selects by label 0, 1, 2
print()
print(obj2[[0, 1, 2]])  # selects by position 0, 1, 2

0    2
1    3
2    1
dtype: int64

a    1
b    2
c    3
dtype: int64


/tmp/ipykernel_18440/2494297800.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(obj2[[0, 1, 2]])  # selects by position 0, 1, 2


In [11]:
# iloc always indexes by position regardless of index type
print(obj1.iloc[[0, 1, 2]])
print()
print(obj2.iloc[[0, 1, 2]])

2    1
0    2
1    3
dtype: int64

a    1
b    2
c    3
dtype: int64


In [12]:
# Label slicing with loc is endpoint inclusive
obj2.loc["b":"c"]

b    2
c    3
dtype: int64

In [13]:
obj2.loc["b":"c"] = 5

obj2

a    1
b    5
c    5
dtype: int64

## DataFrame Indexing with `[]`

`[]` on a DataFrame primarily selects **columns**.

- Single column name → returns a Series.
- List of column names → returns a DataFrame.

### Special Cases

| What you pass | What you get |
|---------------|--------------|
| Single label or list of labels | Column(s) as Series or DataFrame |
| Slice (e.g. `[:2]`) | First 2 **rows** |
| Boolean Series/array | Filtered **rows** |
| Boolean DataFrame | Mask — use for scalar assignment |

## Key Points

- `df[col]` returns a column; `df[[col1, col2]]` returns multiple columns.
- `df[:2]` slices **rows**, not columns — a convenience shortcut.
- `df[bool_array]` filters **rows** where the condition is `True`.
- `df[bool_df] = value` assigns a scalar to every `True` position in the DataFrame.

In [14]:
data = pd.DataFrame(
    np.arange(16).reshape((4, 4)),
    index=["Ohio", "Colorado", "Utah", "New York"],
    columns=["one", "two", "three", "four"]
)

data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [15]:
data["two"]

Ohio         1
Colorado     5
Utah         9
New York    13
Name: two, dtype: int64

In [16]:
data[["three", "one"]]

,three,one
Ohio,2,0
Colorado,6,4
Utah,10,8
New York,14,12


In [17]:
data[:2]

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7


In [18]:
data[data["three"] > 5]

,one,two,three,four
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [19]:
data < 5

,one,two,three,four
Ohio,True,True,True,True
Colorado,True,False,False,False
Utah,False,False,False,False
New York,False,False,False,False


In [20]:
data[data < 5] = 0

data

,one,two,three,four
Ohio,0,0,0,0
Colorado,0,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


## DataFrame Selection with `loc` and `iloc`

`loc` and `iloc` allow selecting both rows and columns simultaneously using comma-separated selectors.

```python
df.loc[row_selector, col_selector]
df.iloc[row_selector, col_selector]
```

- Single label/integer → returns a Series.
- List of labels/integers → returns a DataFrame.
- Slice → returns a subset (label slices with `loc` are **inclusive**).
- Boolean arrays work with `loc` but **not** `iloc`.

### Summary of Indexing Options

| Expression | Selects |
|------------|---------|
| `df[col]` | Single column or list of columns |
| `df.loc[rows]` | Row(s) by label |
| `df.loc[:, cols]` | Column(s) by label |
| `df.loc[rows, cols]` | Rows and columns by label |
| `df.iloc[rows]` | Row(s) by integer position |
| `df.iloc[:, cols]` | Column(s) by integer position |
| `df.iloc[rows, cols]` | Rows and columns by integer position |
| `df.at[row, col]` | Single scalar value by label |
| `df.iat[row, col]` | Single scalar value by integer position |

## Key Points

- `loc` uses labels for both rows and columns.
- `iloc` uses integer positions for both rows and columns.
- Separate row and column selectors with a comma.
- Slices, single values, and lists all work as selectors.
- Boolean arrays can be used with `loc` but not `iloc`.

In [21]:
data.loc["Colorado"]

one      0
two      5
three    6
four     7
Name: Colorado, dtype: int64

In [22]:
data.loc[["Colorado", "New York"]]

,one,two,three,four
Colorado,0,5,6,7
New York,12,13,14,15


In [23]:
data.loc["Colorado", ["two", "three"]]

two      5
three    6
Name: Colorado, dtype: int64

In [24]:
data.iloc[2]

one       8
two       9
three    10
four     11
Name: Utah, dtype: int64

In [25]:
data.iloc[[2, 1]]

,one,two,three,four
Utah,8,9,10,11
Colorado,0,5,6,7


In [26]:
data.iloc[2, [3, 0, 1]]

four    11
one      8
two      9
Name: Utah, dtype: int64

In [27]:
data.iloc[[1, 2], [3, 0, 1]]

,four,one,two
Colorado,7,0,5
Utah,11,8,9


In [28]:
data.loc[:"Utah", "two"]

Ohio        0
Colorado    5
Utah        9
Name: two, dtype: int64

In [29]:
data.iloc[:, :3][data.three > 5]

,one,two,three
Colorado,0,5,6
Utah,8,9,10
New York,12,13,14


In [30]:
data.loc[data.three >= 2]

,one,two,three,four
Colorado,0,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


## Integer Indexing Pitfalls

When a Series has an **integer index**, `[]` can behave unexpectedly:

- pandas refuses to guess whether an integer means a label or a position.
- `ser[-1]` raises a `KeyError` if the index contains integers, because `-1` is not a valid label.
- With a non-integer index, `ser[-1]` falls back to position-based access and works fine.

### Integer Slicing

Slicing with integers is **always position-based**, even on integer-labeled indexes:

```python
ser[:2]  # always first 2 rows by position
```

### Best Practice

Always use `loc` or `iloc` to eliminate ambiguity:

- `ser.loc[label]` — unambiguously label-based.
- `ser.iloc[n]` — unambiguously position-based.

## Key Points

- `[]` on an integer-indexed Series treats integers as labels.
- `ser[-1]` raises `KeyError` on an integer index because `-1` is not a valid label.
- Integer slicing (`ser[:2]`) is always position-based.
- Use `loc` and `iloc` to always get predictable behavior.

In [31]:
ser = pd.Series(np.arange(3.))

ser

0    0.0
1    1.0
2    2.0
dtype: float64

In [32]:
# ser[-1]  # raises KeyError — -1 is not a label in this integer index

# Non-integer index: -1 falls back to position-based and works
ser2 = pd.Series(np.arange(3.), index=["a", "b", "c"])
ser2[-1]

/tmp/ipykernel_18440/1272817740.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ser2[-1]


np.float64(2.0)

In [33]:
ser.iloc[-1]

np.float64(2.0)

In [34]:
ser[:2]  # integer slicing is always position-based

0    0.0
1    1.0
dtype: float64

## Chained Indexing Pitfalls

**Chained indexing** means applying two `[]` operations back-to-back:

```python
df[condition]["col"] = value  # chained — do NOT use for assignment
```

### Why It Fails

- The first `[]` may return a **copy** of the data, not the original DataFrame.
- Assigning to a copy does not modify the original.
- pandas raises a `SettingWithCopyWarning` to alert you.

### The Fix

Use a **single `loc` operation** with both row and column selectors:

```python
df.loc[condition, "col"] = value  # correct
```

### In-Place Assignment with `loc` and `iloc`

Single `loc`/`iloc` operations can safely modify a DataFrame in place:

```python
df.loc[:, "col"] = value     # assign to entire column
df.iloc[2] = value            # assign to entire row
df.loc[df["col"] > n] = value # assign to filtered rows
```

## Key Points

- Chained indexing (`df[...][...] = value`) may silently fail to modify the original.
- `SettingWithCopyWarning` signals that you are assigning to a copy.
- Always use a single `loc` operation for conditional assignment.
- `loc` and `iloc` can safely perform in-place modifications.

In [35]:
data.loc[:, "one"] = 1

data

,one,two,three,four
Ohio,1,0,0,0
Colorado,1,5,6,7
Utah,1,9,10,11
New York,1,13,14,15


In [36]:
data.iloc[2] = 5

data

,one,two,three,four
Ohio,1,0,0,0
Colorado,1,5,6,7
Utah,5,5,5,5
New York,1,13,14,15


In [37]:
data.loc[data["four"] > 5] = 3

data

,one,two,three,four
Ohio,1,0,0,0
Colorado,3,3,3,3
Utah,5,5,5,5
New York,3,3,3,3


In [38]:
# Chained indexing — may trigger SettingWithCopyWarning and NOT modify data
# data.loc[data.three == 5]["three"] = 6

# Correct: single loc operation with row and column selector
data.loc[data.three == 5, "three"] = 6

data

,one,two,three,four
Ohio,1,0,0,0
Colorado,3,3,3,3
Utah,5,5,6,5
New York,3,3,3,3
